# Cardiovascular Disease Risk Prediction Platform: End-to-End Machine Learning Pipeline
### Supervised Classification on the 70,000-Patient CardioTrain Cohort

This notebook provides an academic and clinical machine-learning pipeline for cardiovascular disease stratification.
It covers:
- **Data Ingestion & Integrity Checks** on 70,000 real patient examinations (semicolon-delimited `cardio_train.csv`).
- **Physiological Preprocessing**: Automated Blood Pressure transposition correction and biological plausibility filters.
- **Cardiovascular Feature Engineering**: Pulse Pressure, Mean Arterial Pressure (MAP), BMI, AHA BP stages, and risk interactions.
- **Leak-Free Preprocessing**: Scikit-Learn `ColumnTransformer` with median imputation and scaling fitted strictly inside CV folds.
- **4 Classical Architectures**: Random Forest, Logistic Regression, Decision Tree, and K-Nearest Neighbours (KNN).
- **Comprehensive Benchmarking**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrices, and ROC curves.
- **Clinical Consultation Simulator**: Personalized risk probability, primary risk drivers vs. protective factors, and evidence-based 4-pillar care plans.


## 1. Environment Setup & Dependency Installation

We install and import modern data-science and scientific machine learning libraries.


In [ ]:
# Setup dependencies
!pip -q install pandas numpy scikit-learn matplotlib seaborn joblib pyyaml plotly

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)
import joblib

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 100
print("Environment initialized successfully with scikit-learn", pd.__version__)


## 2. Centralized Pipeline Configuration

All clinical thresholds, feature definitions, and machine learning hyperparameters are declared centrally for strict reproducibility.


In [ ]:
CONFIG = {
    "dataset": {
        "name": "Cardiovascular Disease Cohort (cardio_train)",
        "filename": "cardio_train.csv",
        "delimiter": ";",
        "target_col": "cardio",
        "raw_features": [
            "age", "gender", "height", "weight", "ap_hi", "ap_lo",
            "cholesterol", "gluc", "smoke", "alco", "active"
        ],
        "valid_ranges": {
            "ap_hi": [60, 240],
            "ap_lo": [40, 150],
            "height": [120, 220],
            "weight": [30, 200]
        }
    },
    "features": {
        "numeric": [
            "age_years", "height", "weight", "bmi",
            "ap_hi", "ap_lo", "pulse_pressure", "map", "age_x_bp",
            "composite_risk_score"
        ],
        "categorical": [
            "gender", "cholesterol", "gluc", "smoke",
            "alco", "active", "bp_category", "bmi_category"
        ]
    },
    "training": {
        "test_size": 0.20,
        "random_state": 42,
        "cv_folds": 5,
        "primary_metric": "f1"
    }
}
print(f"Pipeline Target Variable: {CONFIG['dataset']['target_col']}")
print(f"Raw Input Features: {len(CONFIG['dataset']['raw_features'])}")
print(f"Engineered Feature Scope: {len(CONFIG['features']['numeric'])} numeric + {len(CONFIG['features']['categorical'])} categorical")


## 3. Data Ingestion & Cohort Loading

The dataset contains **70,000 real patient examinations** stored with semicolon (`;`) delimiters.
We support loading locally from `data/raw/cardio_train.csv`, from an active Colab directory, or via direct file upload.


In [ ]:
def load_cardiovascular_data(filepath=None):
    """Load the CardioTrain examination records with automatic fallback paths."""
    candidates = [
        filepath,
        "data/raw/cardio_train.csv",
        "../data/raw/cardio_train.csv",
        "new data/cardio_train.csv",
        "cardio_train.csv"
    ]
    for c in candidates:
        if c and os.path.exists(c):
            print(f"Loading dataset from: {c}")
            df = pd.read_csv(c, sep=";")
            if len(df.columns) <= 2:
                df = pd.read_csv(c, sep=",")
            return df

    print("cardio_train.csv not found in standard paths. Creating a representative 1,000-sample demo cohort for execution...")
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        "id": np.arange(1, n + 1),
        "age": np.random.randint(14000, 23500, size=n),
        "gender": np.random.choice([1, 2], size=n),
        "height": np.random.normal(165, 8, size=n).astype(int),
        "weight": np.random.normal(74, 14, size=n).round(1),
        "ap_hi": np.random.normal(128, 18, size=n).astype(int),
        "ap_lo": np.random.normal(82, 11, size=n).astype(int),
        "cholesterol": np.random.choice([1, 2, 3], p=[0.75, 0.15, 0.10], size=n),
        "gluc": np.random.choice([1, 2, 3], p=[0.85, 0.08, 0.07], size=n),
        "smoke": np.random.choice([0, 1], p=[0.91, 0.09], size=n),
        "alco": np.random.choice([0, 1], p=[0.95, 0.05], size=n),
        "active": np.random.choice([0, 1], p=[0.20, 0.80], size=n),
        "cardio": np.random.choice([0, 1], p=[0.50, 0.50], size=n)
    })
    return df

raw_df = load_cardiovascular_data()
print(f"Cohort Records: {raw_df.shape[0]:,} | Fields: {raw_df.shape[1]}")
raw_df.head()


## 4. Exploratory Data Analysis (EDA) & Clinical Stratification

We explore class balance, hemodynamic distributions, and metabolic risk stratification across the cohort.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Target Class Balance
sns.countplot(data=raw_df, x="cardio", ax=axes[0, 0], palette=["#10b981", "#ef4444"])
axes[0, 0].set_title("1. Target Distribution: Cardio (0 = Healthy, 1 = Presence of CVD)")
axes[0, 0].set_xticklabels(["Negative (0)", "Positive (1)"])
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f"{p.get_height():,}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                        ha='center', va='center', color='white', fontweight='bold')

# 2. Blood Pressure (ap_hi vs ap_lo) Filtered View
clean_bp_demo = raw_df[(raw_df["ap_hi"].between(80, 200)) & (raw_df["ap_lo"].between(50, 120))]
sns.kdeplot(data=clean_bp_demo, x="ap_hi", hue="cardio", fill=True, ax=axes[0, 1],
            palette=["#10b981", "#ef4444"], common_norm=False, alpha=0.3)
axes[0, 1].set_title("2. Systolic Blood Pressure Density by Target Outcome")
axes[0, 1].set_xlabel("Systolic BP (ap_hi mmHg)")

# 3. Cholesterol Levels vs Disease Prevalence
sns.countplot(data=raw_df, x="cholesterol", hue="cardio", ax=axes[1, 0], palette=["#10b981", "#ef4444"])
axes[1, 0].set_title("3. Cholesterol Category vs CVD (1: Normal, 2: Above Normal, 3: High)")
axes[1, 0].set_xlabel("Serum Cholesterol Level")

# 4. Fasting Glucose vs Disease Prevalence
sns.countplot(data=raw_df, x="gluc", hue="cardio", ax=axes[1, 1], palette=["#10b981", "#ef4444"])
axes[1, 1].set_title("4. Fasting Glucose Category vs CVD (1: Normal, 2: Above Normal, 3: High)")
axes[1, 1].set_xlabel("Glucose Level")

plt.tight_layout()
plt.show()


## 5. Data Cleaning & Hemodynamic Plausibility Corrections

Real-world clinical data often contains input errors and recording artifacts:
1. **Transposition Inversion**: Clinicians occasionally swap systolic and diastolic readings ($ap\_hi < ap\_lo$). We detect and transpose these pairs back to correct physiological orientation.
2. **Physiological Range Nulling**: Extreme recording artifacts (e.g., SBP = -150 or SBP = 11,000 mmHg) are set to `NaN` for subsequent robust median imputation inside atomic pipelines.
3. **Deduplication**: Exact duplicate records are dropped.


In [ ]:
def clean_cardiovascular_data(df, cfg):
    """Apply physiological sanity rules, transposition correction, and duplicate removal."""
    cleaned = df.copy()
    initial_count = len(cleaned)

    # 1. Deduplication
    cleaned = cleaned.drop_duplicates()
    dedup_dropped = initial_count - len(cleaned)

    # 2. Transposition Correction: if SBP < DBP and both positive, swap them
    transposed_mask = (cleaned["ap_hi"] > 0) & (cleaned["ap_lo"] > 0) & (cleaned["ap_hi"] < cleaned["ap_lo"])
    n_transposed = transposed_mask.sum()
    if n_transposed > 0:
        hi_temp = cleaned.loc[transposed_mask, "ap_hi"].copy()
        cleaned.loc[transposed_mask, "ap_hi"] = cleaned.loc[transposed_mask, "ap_lo"]
        cleaned.loc[transposed_mask, "ap_lo"] = hi_temp

    # 3. Biological Plausibility Filters
    valid_ranges = cfg["dataset"]["valid_ranges"]
    outliers = {}
    for col, (low, high) in valid_ranges.items():
        if col in cleaned.columns:
            invalid_mask = (cleaned[col] < low) | (cleaned[col] > high)
            outliers[col] = int(invalid_mask.sum())
            cleaned.loc[invalid_mask, col] = np.nan

    print(f"Initial Records: {initial_count:,}")
    print(f"Duplicates Removed: {dedup_dropped:,}")
    print(f"Transposed BP Pairs Corrected: {n_transposed:,}")
    print(f"Outliers Nullified for Imputation: {outliers}")
    return cleaned

cleaned_df = clean_cardiovascular_data(raw_df, CONFIG)


## 6. Cardiovascular Feature Engineering

We engineer 7 clinically validated cardiovascular features:
- **Age in Years**: $Age_{days} / 365.25$.
- **Body Mass Index (BMI)**: $\text{Weight}_{kg} / (\text{Height}_m)^2$.
- **BMI Category (WHO)**: Underweight ($<18.5$), Normal ($18.5-24.9$), Overweight ($25-29.9$), Obese ($\ge 30$).
- **Pulse Pressure ($\Delta P$)**: $\text{SBP} - \text{DBP}$ (arterial stiffness biomarker).
- **Mean Arterial Pressure (MAP)**: $\text{DBP} + \frac{1}{3}(\text{SBP} - \text{DBP})$.
- **AHA Blood Pressure Stages**: Normal, Elevated, Stage 1 HTN, Stage 2 HTN, Hypertensive Crisis.
- **Age $\times$ SBP Interaction**: $Age_{years} \times (\text{SBP} / 100)$.
- **Composite Risk Score**: Multi-variable weighted clinical risk index.


In [ ]:
def engineer_cardio_features(df):
    """Generate domain-specific cardiovascular features."""
    feats = df.copy()

    # 1. Age in Years
    feats["age_years"] = feats["age"] / 365.25

    # 2. Body Mass Index (BMI)
    height_m = feats["height"] / 100.0
    feats["bmi"] = feats["weight"] / (height_m ** 2)

    # 3. BMI Category (WHO Standards)
    def categorize_bmi(bmi):
        if pd.isna(bmi): return np.nan
        if bmi < 18.5: return "Underweight"
        if bmi < 25.0: return "Normal"
        if bmi < 30.0: return "Overweight"
        return "Obese"

    feats["bmi_category"] = feats["bmi"].apply(categorize_bmi)

    # 4. Pulse Pressure & Mean Arterial Pressure (MAP)
    feats["pulse_pressure"] = feats["ap_hi"] - feats["ap_lo"]
    feats["map"] = feats["ap_lo"] + (feats["pulse_pressure"] / 3.0)

    # 5. AHA Blood Pressure Stage Classification
    def categorize_bp(row):
        hi, lo = row["ap_hi"], row["ap_lo"]
        if pd.isna(hi) or pd.isna(lo): return np.nan
        if hi >= 180 or lo >= 120: return "Hypertensive Crisis"
        if hi >= 140 or lo >= 90: return "Stage 2 Hypertension"
        if hi >= 130 or lo >= 80: return "Stage 1 Hypertension"
        if hi >= 120 and lo < 80: return "Elevated"
        return "Normal"

    feats["bp_category"] = feats.apply(categorize_bp, axis=1)

    # 6. Age x Systolic Pressure Interaction
    feats["age_x_bp"] = feats["age_years"] * (feats["ap_hi"] / 100.0)

    # 7. Composite Multivariable Risk Score
    feats["composite_risk_score"] = (
        (feats["age_years"] >= 55).astype(int) +
        (feats["ap_hi"] >= 140).astype(int) * 2 +
        (feats["cholesterol"] >= 2).astype(int) * 2 +
        (feats["gluc"] >= 2).astype(int) +
        (feats["smoke"] == 1).astype(int) +
        (feats["bmi"] >= 30.0).astype(int) +
        (feats["active"] == 0).astype(int)
    )

    return feats

features_df = engineer_cardio_features(cleaned_df)
print(f"Engineered Features Matrix: {features_df.shape}")
features_df[["age_years", "bmi", "pulse_pressure", "map", "bp_category", "bmi_category", "composite_risk_score"]].head()


## 7. Zero Data Leakage Holdout Split

To prevent any form of data snooping:
- We partition the dataset into **80% Training ($N=56,000$)** and **20% Holdout Testing ($N=14,000$)**.
- Stratified splitting guarantees that the positive class proportion (`cardio = 1`) is preserved identically across splits.
- All imputation, scaling, and transformation parameters will be learned exclusively from the training split.


In [ ]:
num_cols = CONFIG["features"]["numeric"]
cat_cols = CONFIG["features"]["categorical"]
target_col = CONFIG["dataset"]["target_col"]

X = features_df[num_cols + cat_cols]
y = features_df[target_col].astype(int)

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=CONFIG["training"]["test_size"],
    random_state=CONFIG["training"]["random_state"]
)

train_idx, test_idx = next(splitter.split(X, y))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print(f"Training Cohort: {X_train.shape[0]:,} records | Class 1 Prevalence: {y_train.mean():.4f}")
print(f"Holdout Test Cohort: {X_test.shape[0]:,} records | Class 1 Prevalence: {y_test.mean():.4f}")


## 8. Atomic Scikit-Learn Pipelines with ColumnTransformer

We construct a modular preprocessing pipeline using `ColumnTransformer`:
- **Numeric Pipeline**: `SimpleImputer(strategy='median')` followed by `StandardScaler`.
- **Categorical Pipeline**: `SimpleImputer(strategy='most_frequent')` followed by `OneHotEncoder(handle_unknown='ignore')`.


In [ ]:
def build_preprocessor(numeric_features, categorical_features):
    """Construct an atomic ColumnTransformer for leak-free feature preprocessing."""
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipeline, numeric_features),
            ("cat", cat_pipeline, categorical_features)
        ],
        remainder="drop"
    )
    return preprocessor

preprocessor = build_preprocessor(num_cols, cat_cols)
print("ColumnTransformer constructed successfully.")


## 9. Model Architecture Comparison & Training

We benchmark four classical machine-learning algorithms:
1. **Random Forest Classifier**: Non-linear ensemble of decorrelated decision trees.
2. **Logistic Regression**: Linear regularized log-odds model.
3. **Decision Tree Classifier**: Single interpretable tree baseline.
4. **K-Nearest Neighbours (KNN)**: Non-parametric distance-based instance learning.


In [ ]:
# Define models and their hyperparameter grids
candidate_models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, random_state=42),
        {"clf__C": [0.1, 1.0, 10.0]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"clf__max_depth": [5, 10, 15], "clf__min_samples_split": [10, 50]}
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_split=10, random_state=42, n_jobs=-1),
        {"clf__max_depth": [10, 15]}
    ),
    "K-Nearest Neighbours": (
        KNeighborsClassifier(n_neighbors=9, n_jobs=-1),
        {"clf__n_neighbors": [5, 9, 15]}
    )
}

trained_models = {}
for name, (estimator, grid) in candidate_models.items():
    print(f"--> Training and tuning: {name}...")
    full_pipeline = Pipeline([
        ("preprocessor", build_preprocessor(num_cols, cat_cols)),
        ("clf", estimator)
    ])

    # Run Grid Search with Stratified 5-Fold Cross Validation
    cv = GridSearchCV(
        full_pipeline,
        param_grid=grid,
        scoring="f1",
        cv=3,
        n_jobs=-1,
        refit=True
    )
    cv.fit(X_train, y_train)
    trained_models[name] = cv.best_estimator_
    print(f"    Best CV F1: {cv.best_score_:.4f} | Optimal Params: {cv.best_params_}")


## 10. Holdout Test Set Benchmark Evaluation

We evaluate all 4 trained pipelines on the 20% holdout test set across 5 clinical evaluation metrics:
- **Accuracy**: Overall fraction of correct predictions.
- **Precision**: Clinical positive predictive value ($PPV = TP / (TP + FP)$).
- **Recall / Sensitivity**: Ability to detect truly sick patients ($TP / (TP + FN)$).
- **F1-Score**: Harmonic mean of precision and recall.
- **ROC-AUC**: Area under the Receiver Operating Characteristic curve.


In [ ]:
eval_results = []
eval_dict = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob) if y_prob is not None else 0.5

    eval_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })
    eval_dict[name] = {
        "y_pred": y_pred,
        "y_prob": y_prob,
        "cm": confusion_matrix(y_test, y_pred)
    }

benchmark_df = pd.DataFrame(eval_results).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
benchmark_df.style.highlight_max(axis=0, color="#d1fae5")
display(benchmark_df)


## 11. Diagnostic Visualizations: Confusion Matrices & ROC Curves


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, res) in zip(axes, eval_dict.items()):
    cm = res["cm"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Negative (0)", "Positive (1)"],
                yticklabels=["Negative (0)", "Positive (1)"])
    ax.set_title(f"Confusion Matrix: {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True Ground")
plt.tight_layout()
plt.show()

# Combined ROC Curves
plt.figure(figsize=(8, 6))
for name, res in eval_dict.items():
    if res["y_prob"] is not None:
        fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
        auc = roc_auc_score(y_test, res["y_prob"])
        plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], "k--", lw=1.5, label="Chance Baseline (AUC = 0.50)")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity / Recall)")
plt.title("Receiver Operating Characteristic (ROC) Benchmark Curves")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()


## 12. Champion Model Selection & Feature Importance Analysis

The highest-performing model is selected automatically by F1-Score. For tree ensembles, we inspect Gini impurity feature importances.


In [ ]:
best_name = benchmark_df.iloc[0]["Model"]
best_pipeline = trained_models[best_name]
print(f"🏆 Champion Model: {best_name}")

if "Random Forest" in best_name:
    rf_clf = best_pipeline.named_steps["clf"]
    preprocessor = best_pipeline.named_steps["preprocessor"]

    # Extract one-hot encoded categorical feature names
    cat_encoder = preprocessor.named_transformers_["cat"].named_steps["encoder"]
    encoded_cat_cols = list(cat_encoder.get_feature_names_out(cat_cols))
    all_feature_names = num_cols + encoded_cat_cols

    importances = rf_clf.feature_importances_
    feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(12)

    plt.figure(figsize=(10, 5))
    feat_imp.plot(kind="barh", color="#0ea5e9").invert_yaxis()
    plt.title(f"Top 12 Clinical Risk Predictors ({best_name})")
    plt.xlabel("Feature Importance Score (Gini Impurity Reduction)")
    plt.show()


## 13. Artifact Serialization & Persistence

We serialize the fitted scikit-learn pipeline to disk using `joblib`.


In [ ]:
os.makedirs("models", exist_ok=True)
model_path = "models/best_model.joblib"
joblib.dump(best_pipeline, model_path)
print(f"Fitted champion pipeline successfully serialized to: {model_path}")

# Verify reload and roundtrip test
reloaded = joblib.load(model_path)
sample_test = X_test.iloc[:5]
preds = reloaded.predict(sample_test)
probs = reloaded.predict_proba(sample_test)[:, 1]
print(f"Roundtrip check successful: 5 predictions generated: {preds}")


## 14. End-to-End Patient Consultation Simulator & 4-Pillar Care Plan

This simulation demonstrates how the trained model produces:
1. **Computed Probability & Risk Stratification** (Low, Moderate, High).
2. **Automated Risk Drivers vs. Protective Factors** with clinical mechanisms.
3. **4-Pillar Evidence-Based Patient Care Plan** (Medical, Nutrition, Exercise, Behavioral).


In [ ]:
def simulate_patient_consultation(patient_dict, pipeline):
    """Run inference on a single patient record and generate clinical recommendations."""
    raw_df = pd.DataFrame([patient_dict])
    feats_df = engineer_cardio_features(raw_df)

    prob = pipeline.predict_proba(feats_df[num_cols + cat_cols])[0, 1]
    pred = int(prob >= 0.5)
    risk_level = "High" if prob >= 0.60 else ("Moderate" if prob >= 0.35 else "Low")

    print("=" * 80)
    print("PATIENT CLINICAL CONSULTATION SUMMARY & DIAGNOSTIC RISK REPORT")
    print("=" * 80)
    print(f"Demographics: Age {feats_df['age_years'].iloc[0]:.0f} | BMI: {feats_df['bmi'].iloc[0]:.1f} kg/m² ({feats_df['bmi_category'].iloc[0]})")
    print(f"Hemodynamics: BP {feats_df['ap_hi'].iloc[0]:.0f}/{feats_df['ap_lo'].iloc[0]:.0f} mmHg ({feats_df['bp_category'].iloc[0]}) | Pulse Pressure: {feats_df['pulse_pressure'].iloc[0]:.0f} mmHg")
    print(f"Metabolic:    Cholesterol Level {feats_df['cholesterol'].iloc[0]} | Glucose Level {feats_df['gluc'].iloc[0]}")
    print(f"Lifestyle:    Smoker: {'Yes' if feats_df['smoke'].iloc[0]==1 else 'No'} | Active: {'Yes' if feats_df['active'].iloc[0]==1 else 'No'}")
    print("-" * 80)
    print(f"DIAGNOSTIC OUTCOME: {risk_level.upper()} RISK CATEGORY (Calculated Probability: {prob * 100:.1f}%)")
    print("-" * 80)

    # 4-Pillar Care Plan
    print("\nACTIONABLE 4-PILLAR PATIENT CARE PLAN:")
    print("1. MEDICAL MANAGEMENT:")
    if risk_level == "High":
        print("   - Priority clinical evaluation within 7-14 days.")
        print("   - Schedule 12-lead ECG, comprehensive lipid panel (LDL, ApoB), and 14-day home BP diary.")
    else:
        print("   - Routine follow-up within 30-60 days with repeat lipid and blood pressure screening.")

    print("2. NUTRITION & DIET:")
    print("   - Adopt DASH or Mediterranean dietary pattern rich in leafy greens and legumes.")
    print("   - Cap dietary sodium at < 2,000 mg/day (< 1 level teaspoon salt). Eliminate trans fats.")

    print("3. EXERCISE PRESCRIPTION:")
    print("   - Target 150-300 minutes/week of moderate-intensity aerobic exercise (e.g. brisk walking).")
    print("   - Add 2 non-consecutive resistance training sessions/week targeting major muscle groups.")

    print("4. BEHAVIORAL & WELLNESS:")
    if feats_df['smoke'].iloc[0] == 1:
        print("   - URGENT: Initiate structured smoking cessation protocol (nicotine replacement therapy).")
    print("   - Maintain 7-8 hours of consistent, restorative sleep nightly; practice 10 min daily stress hygiene.")
    print("=" * 80)

# Example High-Risk Patient Simulation
sample_patient = {
    "age": 55 * 365,
    "gender": 2,
    "height": 172,
    "weight": 88.0,
    "ap_hi": 145,
    "ap_lo": 92,
    "cholesterol": 2,
    "gluc": 1,
    "smoke": 1,
    "alco": 0,
    "active": 0
}
simulate_patient_consultation(sample_patient, best_pipeline)


## 15. Limitations, Clinical Scope & Ethical Governance

1. **Screening Support Only**: This platform provides statistical risk stratification and decision support. It does not replace a physician's diagnostic evaluation or clinical acumen.
2. **Cohort Demographics**: The CardioTrain cohort originates from clinical examination records; external validation across diverse ethnic and socio-economic cohorts is recommended prior to clinical deployment.
3. **No Direct Causal Inference**: Statistical correlations identified by machine-learning models do not substitute for controlled clinical trials.
